# AKROMEGALİ KOHORTUNDA MORTALİTE TAHMİNİ

**Veri Bilimi Uygulamaları · Kasım 2026**

Keşifsel veri analizi • Veri kalitesi • Sızıntı denetimi • Makine öğrenmesi • Model yorumlama

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from IPython.display import display, Image
ROOT = next((p for p in [Path.cwd(), Path.cwd().parent] if (p/'run_project.py').exists()), None)
if ROOT is None:
    raise FileNotFoundError('run_project.py bulunamadı: Notebooku repo klasöründe açın.')
DATA = ROOT/'akromegali_alistirma.csv'
df = pd.read_csv(DATA)
cv_results = pd.read_csv(ROOT/'reports/cv_summary_from_mlflow.csv')
test_results = pd.read_csv(ROOT/'reports/test_summary_from_mlflow.csv')
print('Proje klasörü:', ROOT)
print('Boyut:', df.shape)

# 01 / Hedef dağılımı

Dengesiz hedef dağılımı nedeniyle **accuracy tek başına** değerlendirme için yeterli değildir.

In [ ]:
counts = df['Alive'].value_counts().reindex([0,1],fill_value=0)
fig,ax=plt.subplots(figsize=(7,4))
ax.bar(['Vefat','Hayatta'],counts.values,color=['#d46c63','#2a6b99'])
ax.set_ylabel('Hasta sayısı')
for i,n in enumerate(counts.values): ax.text(i,n+4,str(n),ha='center')
plt.show()

# 02 / Eksik veri analizi

Eksik değer oranları sütunlara göre farklılaşıyor. Satırları topluca silmek az sayıdaki vefat gözlemini kaybettirebilir.

In [ ]:
missing=df.isna().mean().sort_values(ascending=False).head(10)*100
display(missing.rename('Eksik (%)').round(1).to_frame())
fig,ax=plt.subplots(figsize=(8,4))
missing.sort_values().plot.barh(ax=ax,color='#457b9d')
ax.set_xlabel('Eksik değer (%)')
plt.tight_layout();plt.show()

# 03 / Veri sözleşmesi

Hedefin ikili olması, yaş sıralaması ve ameliyat/remisyon alanlarının tutarlılığı kontrol edilir.

İhlal saptanan ve eksiklik nedeniyle kontrol edilemeyen satırlar ayrı değerlendirilir.

In [ ]:
# Üç kontrolün gerçek sonuçlarını yeniden üretmek için:
import subprocess,sys
print('Tam veri sözleşmesi + model eğitimi için:')
print(f'{sys.executable} {ROOT/"run_project.py"} --no-mlflow')
print('Betik çalışınca results/data_contract.csv oluşur.')

# 04 / Veri sızıntısı: tahmin anı

Takip sonu, ameliyat sonrası ve sonuç değişkenleri ana modelden çıkarıldı.

**Sınırlılık:** Ameliyat öncesi ölçümlerin tamamının seçilen tahmin anında mevcut olduğu bağımsız biçimde doğrulanmış değil.

In [ ]:
FEATURES=['Age_in_Diagnosis','Gender','Preop_GH','Preop_IGF_1','Preop_Tm_Size_MAX_mm']
y=(df['Alive']==0).astype(int)
X=df[FEATURES].copy()
print('Model girdileri:', FEATURES)
print('Pozitif sınıf (vefat) oranı:', round(y.mean(),4))

# 05 / Ön işleme ve sınıflandırıcılar

Sayısal eksiklikler medyanla doldurulur; eksiklik göstergeleri eklenir. Lojistik regresyonda ölçekleme, cinsiyette kategorik kodlama kullanılır.

İşlemler **Pipeline** içinde uygulanır.

In [ ]:
from run_project import models
est=models()
for name,model in est.items():
    print(name, ':', type(model).__name__)

# 06 / Eğitim-test ayrımı

**%80 eğitim / %20 test**, stratified, random_state=42.

Ön işleme adımları eğitim setinden öğrenilir; çapraz doğrulama yalnızca eğitim seti üzerinde gerçekleştirilir.

In [ ]:
from sklearn.model_selection import train_test_split
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.20,random_state=42,stratify=y)
print(f'Train: {len(ytr)} hasta, {int(ytr.sum())} vefat')
print(f'Test: {len(yte)} hasta, {int(yte.sum())} vefat')

# 07 / Tekrarlı çapraz doğrulama

**RepeatedStratifiedKFold:** 5 kat × 5 tekrar, eğitim verisi üzerinde.

Lojistik regresyonun ortalama ROC-AUC ve AP değerleri diğer iki modelden yüksektir.

In [ ]:
display(cv_results[['Model','roc_auc_mean','pr_auc_mean','recall_death_mean','f1_death_mean']].round(4))
display(Image(filename=str(ROOT/'model_comparison_cv.png'),width=850))

# 08 / Ayrılmış test sonuçları

Testte yalnızca **10 vefat** bulunduğundan performans sayıları yüksek belirsizlik taşır. Bu test kümesi geliştirme sırasında daha önce incelenmiştir.

In [ ]:
display(test_results[['Model','accuracy','roc_auc','pr_auc','precision_death','recall_death','f1_death']].round(4))

# 09 / Genel özellik etkileri

Lojistik regresyonda tanı yaşı, tümör boyutu, GH ve IGF-1 ile eksiklik göstergeleri öne çıkıyor.

Katsayılar **nedensel etki** değildir.

In [ ]:
display(Image(filename=str(ROOT/'global_feature_importance.png'),width=900))

# 10 / Yanlış pozitif olgu ve güvenilirlik

Orijinal analizde **hasta #240**, hayatta olmasına rağmen vefat sınıfında tahmin edilmiştir. Tanı yaşı ve tümör boyutunun eksikliği yerel açıklamada dikkat çekmiştir.

Sınıf ağırlıklı model skorları kalibre klinik risk olarak yorumlanmamalıdır.

In [ ]:
from sklearn.metrics import confusion_matrix
print('Yanlış pozitif ayrıntıları: run_project.py çalıştırıldıktan sonra')
print('results/selected_false_positive.json ve results/local_contributions.csv')

# 11 / Sonuç

Tekrarlı çapraz doğrulamada **Logistic Regression**, ROC-AUC **0,8205** ve PR-AUC (AP) **0,4043** ile en güçlü model adayıdır. Modelin değerlendirdiği önemli özellikler tanı yaşı, tümör boyutu, GH ve IGF-1 ile bazı eksiklik göstergeleridir. Testte vefat sınıfı recall **0,70** olsa da precision **0,175** düzeyindedir. Az sayıdaki vefat, eksiklikler, zaman sızıntısı ihtimali ve daha önce incelenmiş test kümesi sonuçları sınırlar. Bu nedenle bulgular keşifsel niteliktedir ve bağımsız doğrulama olmadan klinik karar amacıyla kullanılmamalıdır.